# DeepWeb Dataset Builder: High-Integrity Pretraining Curation

This notebook ingests raw or educational web data (e.g., `HuggingFaceFW/fineweb-edu`), runs the **Discrete Evidence Engine (DEE)** compiled in Rust for mathematical and epistemic verification, and outputs the **DeepWeb** pretraining dataset partitioned into the **SmolLM2 4-stage developmental curriculum**:
- **Stage 1 (Fundamental)**: Foundational narrative & simple encyclopedic concepts
- **Stage 2 (Intermediate)**: High school & explanatory STEM/humanities
- **Stage 3 (Advanced)**: Undergraduate textbooks, legal codes & technical specifications
- **Stage 4 (Research)**: Peer-reviewed research, formal mathematical proofs & patents

In [ ]:
# 1. Install prerequisites
!pip install -q datasets pyarrow pandas

In [ ]:
# 2. Locate and stage DEE standalone binary into working directory
import os, sys, shutil, glob, subprocess

print("Available input directories:", glob.glob("/kaggle/input/*"))
candidates = glob.glob("/kaggle/input/**/dee", recursive=True) + glob.glob("../input/**/dee", recursive=True) + glob.glob("./dee")
print("Found binary candidates:", candidates)

dee_bin = "/kaggle/working/dee"
if candidates:
    src_bin = candidates[0]
    if os.path.abspath(src_bin) != os.path.abspath(dee_bin):
        shutil.copy2(src_bin, dee_bin)
    os.chmod(dee_bin, 0o755)
    print(f"Successfully staged executable DEE binary: {dee_bin}")
else:
    print("Listing all files in inputs:", glob.glob("/kaggle/input/**/*", recursive=True))
    raise FileNotFoundError("DEE binary not found in /kaggle/input")

!{dee_bin} --help


In [ ]:
# 3. Locate builder script and execute Full Scale DeepWeb Parquet Ingestion
import os, glob

builder_candidates = glob.glob("/kaggle/input/**/deepweb_builder.py", recursive=True) + glob.glob("../input/**/deepweb_builder.py", recursive=True) + glob.glob("./deepweb_builder.py")
if builder_candidates:
    builder_script = builder_candidates[0]
else:
    builder_script = "deepweb_builder.py"
print(f"Using builder script: {builder_script}")

# Direct compressed Parquet streaming with 18.5GB disk limit safety guard
!python3 {builder_script} --dataset HuggingFaceFW/fineweb-edu --subset sample-10BT --chunk-size 10000 --disk-limit-gb 18.5 --min-free-gb 1.5 --output-dir /kaggle/working/deepweb --dee-binary {dee_bin}


In [ ]:
# 4. Summary of Generated Parquet Curriculum Partitions
from pathlib import Path
import pandas as pd

parquet_root = Path("/kaggle/working/deepweb/parquet_shards")
print("=" * 75)
print("DEEPWEB PARQUET PRETRAINING DATASET READY")
print("=" * 75)
for sdir in sorted(parquet_root.iterdir()):
    if sdir.is_dir():
        shards = list(sdir.glob("*.parquet"))
        total_bytes = sum(f.stat().st_size for f in shards)
        sample_df = pd.read_parquet(shards[0]) if shards else None
        sample_count = len(sample_df) if sample_df is not None else 0
        print(f"{sdir.name:<26}: {len(shards):>4} shards | {total_bytes / (1024*1024):>8.2f} MB | e.g. shard 0: {sample_count} docs")
print("=" * 75)
